# T24 — real Qwen3-32B numerical freeze

Run this **before any T29 matched-control outcomes are inspected**. PR #48 already froze the control mathematics; this notebook creates the missing genuine calibration activations, runs the reviewed numerical-freeze tool, and persists a small review bundle while the large NPZs stay on Drive.

Use an **80 GB-class CUDA GPU** (A100-80GB, H100-80GB, or G4/RTX PRO 6000 96GB). The run directory is isolated by the exact Git commit and a fail-closed run contract, so interrupted runs may resume only under the same code/environment contract instead of silently reusing stale shards.

If an earlier run completed all 100 capture shards but stopped at the old q25/q75 engagement gate, do **not** rerun Qwen. Pull the latest branch and use Cell 3A to recover the numerical freeze from the existing calibration NPZ under the approved 2026-08-21 pre-outcome deviation.


In [ ]:
# 0. T21-compatible Qwen environment; do not install vLLM or quantization.
import os, sys, subprocess, importlib.metadata as md
from pathlib import Path

REQ = [
    'transformers==4.57.6',
    'accelerate==1.14.0',
    'huggingface_hub==0.36.2',
    'safetensors',
    'sentencepiece',
    'numpy',
    'pytest>=8,<9',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *REQ])

HF_HOME = Path('/content/hf_cache')
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

for p in ['torch', 'transformers', 'accelerate', 'huggingface_hub', 'numpy']:
    try:
        print(p, md.version(p))
    except Exception as e:
        print(p, 'UNKNOWN', e)
print('HF_HOME', HF_HOME)


In [ ]:
# 1. Mount Drive and clone/update the exact implementation branch.
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import base64, getpass, shutil
PROJECT_DIR = Path('/content/[anonymized-repository-name]')
BRANCH = 't24/real-calibration-capture'

TOKEN = os.environ.get('GITHUB_TOKEN', '').strip() or getpass.getpass('GitHub token (hidden): ').strip()
HF_TOKEN = os.environ.get('HF_TOKEN', '').strip() or getpass.getpass('Hugging Face token (hidden): ').strip()
AUTH = base64.b64encode(f'x-access-token:{TOKEN}'.encode()).decode()

def git(args, cwd=None, capture=False):
    return subprocess.run(
        ['git', '-c', f'http.extraHeader=Authorization: Basic {AUTH}', *args],
        cwd=str(cwd) if cwd else None,
        check=True,
        text=True,
        capture_output=capture,
    )

if not (PROJECT_DIR / '.git').exists():
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    git(['clone', 'https://github.com/[Author-B-GitHub]/[anonymized-repository-name].git', str(PROJECT_DIR)])
else:
    git(['fetch', 'origin', '--prune'], cwd=PROJECT_DIR)

git(['checkout', BRANCH], cwd=PROJECT_DIR)
git(['pull', '--ff-only', 'origin', BRANCH], cwd=PROJECT_DIR)

HEAD = git(['rev-parse', 'HEAD'], cwd=PROJECT_DIR, capture=True).stdout.strip()
REMOTE_HEAD = git(['rev-parse', f'origin/{BRANCH}'], cwd=PROJECT_DIR, capture=True).stdout.strip()
assert HEAD == REMOTE_HEAD, f'local branch is not at origin/{BRANCH}: {HEAD} != {REMOTE_HEAD}'
dirty = git(['status', '--porcelain'], cwd=PROJECT_DIR, capture=True).stdout.strip()
assert not dirty, f'working tree must be clean before production capture/recovery:\n{dirty}'

os.environ['HF_TOKEN'] = HF_TOKEN
print('BRANCH', BRANCH)
print('HEAD', HEAD)
print('PASS: exact remote branch checked out with clean working tree')


In [ ]:
# 2. Fail-closed preflight: frozen input, tests, source artifacts, tokenizer serialization,
#    GPU/BF16/storage, and a commit-isolated resume contract.
import hashlib, json, re, shutil
from datetime import datetime, timezone
import torch

MODEL_ID = 'Qwen/Qwen3-32B'
MODEL_REVISION = '9216db5781bf21249d130ec9da846c4624c16137'
ARTIFACT_REPO = 'lu-christina/assistant-axis-vectors'
ARTIFACT_REVISION = '3b3b788432ad33e3a28d9ff08e88a530c0740814'
AXIS_FILE = 'qwen-3-32b/assistant_axis.pt'
AXIS_SHA = 'a207fe7a36563280b7b29010880aa0082bd8e3113c141cb4a2eed6b46c140211'
CAP_FILE = 'qwen-3-32b/capping_config.pt'
CAP_SHA = '6aec1220487473aaeab80b05d5d960ac54b5dd9080b51ac4bb0bbd1f4330db24'
EXPECTED_MANIFEST_SHA = 'd611e904f3b5d47c71e5ab1f3fa1a84ead6cfd5d94dba337f5c3b71aafef7793'
LAYERS = list(range(46, 54))
MAX_NEW_TOKENS = 256

MANIFEST = Path('/content/drive/MyDrive/[anonymized-repository-name]-t21/input/t21_eval_manifest_frozen.jsonl')
WORK_BASE = Path('/content/drive/MyDrive/[anonymized-repository-name]-t24')
WORK = WORK_BASE / f'real_calibration_{HEAD[:12]}'
WORK.mkdir(parents=True, exist_ok=True)

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(8 << 20), b''):
            h.update(b)
    return h.hexdigest()

assert MANIFEST.exists(), f'missing frozen T21 manifest: {MANIFEST}'
assert sha256(MANIFEST) == EXPECTED_MANIFEST_SHA, 'frozen T21 manifest SHA mismatch'
rows = [json.loads(x) for x in MANIFEST.read_text(encoding='utf-8').splitlines() if x.strip()]
assert len(rows) == 100 and len({r['item_id'] for r in rows}) == 100
assert all(isinstance(r.get('messages'), list) and r['messages'] for r in rows)

subprocess.check_call(
    [
        sys.executable, '-m', 'pytest', '-q',
        'tests/test_t24_capping_controls.py',
        'tests/test_t24_calibration_capture.py',
        'tests/test_t24_engagement_matching_deviation.py',
    ],
    cwd=PROJECT_DIR,
)

assert torch.cuda.is_available(), 'CUDA GPU required'
assert torch.cuda.is_bf16_supported(), 'BF16-capable CUDA GPU required'
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
gpu_vram_gib = [torch.cuda.get_device_properties(i).total_memory / (1024 ** 3) for i in range(torch.cuda.device_count())]
total_vram_gib = sum(gpu_vram_gib)
assert total_vram_gib >= 70.0, f'need >=70 GiB total CUDA VRAM; found {total_vram_gib:.1f} GiB across {gpu_names}'
local_free_gib = shutil.disk_usage('/content').free / (1024 ** 3)
assert local_free_gib >= 80.0, f'need >=80 GiB free local disk; found {local_free_gib:.1f} GiB'

probe = WORK / '.write_probe'
probe.write_bytes(b'T24_WRITE_PROBE\n')
assert probe.read_bytes() == b'T24_WRITE_PROBE\n'
probe.unlink()

from huggingface_hub import hf_hub_download
PRECHECK = WORK / 'preflight_source_artifacts'
PRECHECK.mkdir(parents=True, exist_ok=True)
axis_path = Path(hf_hub_download(ARTIFACT_REPO, AXIS_FILE, repo_type='dataset', revision=ARTIFACT_REVISION, token=HF_TOKEN, local_dir=PRECHECK))
cap_path = Path(hf_hub_download(ARTIFACT_REPO, CAP_FILE, repo_type='dataset', revision=ARTIFACT_REVISION, token=HF_TOKEN, local_dir=PRECHECK))
assert sha256(axis_path) == AXIS_SHA, 'released Assistant Axis SHA mismatch'
assert sha256(cap_path) == CAP_SHA, 'released capping config SHA mismatch'

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, token=HF_TOKEN, use_fast=True, trust_remote_code=False)
prompt_token_lengths = []
for i, row in enumerate(rows):
    messages = row['messages']
    rendered = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    direct_ids = tok.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, enable_thinking=False)
    via_ids = tok(rendered, add_special_tokens=False)['input_ids']
    assert list(direct_ids) == list(via_ids), f'chat-template tokenization mismatch at item {i}'
    low = rendered.lower(); marker = '<|im_start|>assistant'; pos = low.rfind(marker); tail = rendered[pos:] if pos >= 0 else rendered
    opens = len(re.findall(r'<think>', tail, flags=re.I)); closes = len(re.findall(r'</think>', tail, flags=re.I))
    assert opens == closes, f'malformed no-thinking assistant prefix at item {i}'
    blocks = re.findall(r'<think>(.*?)</think>', tail, flags=re.I | re.S)
    assert len(blocks) == opens and all(not b.strip() for b in blocks), f'non-empty reasoning prefill at item {i}'
    prompt_token_lengths.append(len(via_ids))
del tok

CONTRACT = {
    'schema_version': 't24-notebook-run-contract/1.0', 'task': 'T24', 'source_git_sha': HEAD, 'branch': BRANCH,
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'dtype': 'bfloat16', 'thinking': False},
    'manifest_sha256': EXPECTED_MANIFEST_SHA, 'n_items': 100, 'layers': LAYERS,
    'decoding': {'do_sample': False, 'max_new_tokens': MAX_NEW_TOKENS, 'use_cache': True},
    'capture_rows': {'natural_L': 'all hook-visible rows during unsteered generate', 'default_L': 'one equal-item mean row per manifest item'},
    'source_artifacts': {'assistant_axis_sha256': AXIS_SHA, 'capping_config_sha256': CAP_SHA, 'artifact_revision': ARTIFACT_REVISION},
    'environment': {'python': sys.version.split()[0], 'torch': md.version('torch'), 'transformers': md.version('transformers'), 'accelerate': md.version('accelerate'), 'huggingface_hub': md.version('huggingface_hub'), 'cuda_runtime': torch.version.cuda, 'gpu_names': gpu_names, 'gpu_vram_gib': [round(x, 3) for x in gpu_vram_gib]},
}
CONTRACT_PATH = WORK / 'RUN_CONTRACT.json'
contract_bytes = (json.dumps(CONTRACT, indent=2, sort_keys=True) + '\n').encode('utf-8')
CONTRACT_SHA = hashlib.sha256(contract_bytes).hexdigest()
if CONTRACT_PATH.exists():
    assert CONTRACT_PATH.read_bytes() == contract_bytes, f'existing run contract differs: {CONTRACT_PATH}'
else:
    tmp = CONTRACT_PATH.with_suffix('.tmp'); tmp.write_bytes(contract_bytes); os.replace(tmp, CONTRACT_PATH)

print('PASS: frozen 100-item manifest + T24 tests')
print('PASS: source artifact hashes + tokenizer serialization')
print('GPU(s):', list(zip(gpu_names, [round(x, 1) for x in gpu_vram_gib])), 'total GiB', round(total_vram_gib, 1))
print('local free GiB:', round(local_free_gib, 1))
print('prompt tokens: min', min(prompt_token_lengths), 'max', max(prompt_token_lengths))
print('WORK:', WORK)
print('RUN_CONTRACT SHA-256:', CONTRACT_SHA)


In [ ]:
# 3. EXPENSIVE: genuine Qwen capture + numerical freeze.
RESULT_DIR = PROJECT_DIR / 'results' / 't24'
FREEZE = RESULT_DIR / 'source_control_freeze.json'
RECEIPT = RESULT_DIR / 't24_calibration_capture_receipt.json'
for p in [FREEZE, RECEIPT]:
    if p.exists(): p.unlink()
subprocess.check_call([sys.executable, 'tools/gpu_cells/t24_capture_qwen_calibration.py', '--manifest', str(MANIFEST), '--work-dir', str(WORK)], cwd=PROJECT_DIR, env={**os.environ, 'HF_TOKEN': HF_TOKEN})
shards = sorted((WORK / 'shards').glob('*.npz'))
assert len(shards) == 100, f'expected 100 completed per-item shards, found {len(shards)}'
assert FREEZE.exists() and RECEIPT.exists(), 'capture completed without both small review JSONs'
print('PASS: expensive capture completed with 100/100 shards')


In [ ]:
# 3A. CPU-only recovery if 100/100 capture completed under the old q25/q75 gate.
# Skip when Cell 3 completed normally and already wrote FREEZE + RECEIPT.
CALIBRATION = WORK / 't24_qwen_calibration.npz'
PARAMS = WORK / 'source_control_parameters.npz'
AXIS_PT = WORK / 'source_artifacts/qwen-3-32b/assistant_axis.pt'
CAP_PT = WORK / 'source_artifacts/qwen-3-32b/capping_config.pt'
RESULT_DIR = PROJECT_DIR / 'results' / 't24'
FREEZE = RESULT_DIR / 'source_control_freeze.json'
RECEIPT = RESULT_DIR / 't24_calibration_capture_receipt.json'
if FREEZE.exists() and RECEIPT.exists():
    print('SKIP: Cell 3 already completed numerical freeze and receipt normally')
else:
    assert CALIBRATION.exists(), f'missing completed calibration NPZ: {CALIBRATION}'
    assert len(list((WORK / 'shards').glob('*.npz'))) == 100, 'recovery requires 100/100 completed capture shards'
    subprocess.check_call([sys.executable, 'tools/recover_t24_numerical_freeze.py', '--calibration-npz', str(CALIBRATION), '--assistant-axis-pt', str(AXIS_PT), '--capping-config-pt', str(CAP_PT), '--out-npz', str(PARAMS), '--out-json', str(FREEZE), '--receipt', str(RECEIPT)], cwd=PROJECT_DIR)
    print('PASS: reused 100/100 Qwen capture; numerical freeze recovered without model reload')


In [ ]:
# 4. Final fail-closed numerical review + persistent small review bundle.
import zipfile
f = json.loads(FREEZE.read_text(encoding='utf-8')); r = json.loads(RECEIPT.read_text(encoding='utf-8'))
assert f['status'] == 'FROZEN_NUMERICAL_CONTROLS'
assert f['deviation']['id'] == 'T24_ENGAGEMENT_MATCHING_2026-08-21'
assert f['deviation']['source_treatment_changed'] is False
assert f['validation']['primary_controls_engagement_matched'] == 'PASS'
assert r['outcome_labels_used'] is False
normal_status = r['status'] == 'REAL_QWEN_CALIBRATION_CAPTURED'
recovery_status = r['status'] == 'REAL_QWEN_CALIBRATION_CAPTURED_AND_FREEZE_RECOVERED'
assert normal_status or recovery_status, r['status']
if normal_status:
    assert f['source_git_sha'] == r['source_git_sha']; capture_git_sha = r['source_git_sha']; freeze_git_sha = f['source_git_sha']
else:
    capture_git_sha = r['capture_source_git_sha']; freeze_git_sha = r['freeze_source_git_sha']
    assert f['source_git_sha'] == freeze_git_sha and r['expensive_capture_reused'] is True
assert r['model']['id'] == MODEL_ID and r['model']['revision'] == MODEL_REVISION
assert r['model']['dtype'] == 'bfloat16' and r['model']['thinking'] is False
assert r['manifest_sha256'] == EXPECTED_MANIFEST_SHA and r['n_items'] == 100 and r['layers'] == LAYERS
tol = f['validation']['engagement_absolute_tolerance']; ortho = f['validation']['orthogonality_tolerance_absolute_cosine']
print('L  source   random   sphere   |cos(axis,random)|')
for L in LAYERS:
    x = f['layers'][str(L)]
    assert abs(x['abs_cos_axis_random']) <= ortho
    assert abs(x['random_minus_source_engagement']) <= tol and abs(x['sphere_minus_source_engagement']) <= tol
    print(L, f"{x['source_axis_engagement_rate']:.4f}", f"{x['random_engagement_rate']:.4f}", f"{x['sphere_engagement_rate']:.4f}", f"{x['abs_cos_axis_random']:.3e}")
CALIBRATION = WORK / 't24_qwen_calibration.npz'; PARAMS = WORK / 'source_control_parameters.npz'
assert sha256(CALIBRATION) == r['calibration_npz_sha256']
assert sha256(PARAMS) == r['numerical_controls_npz_sha256'] == f['numerical_controls_npz_sha256']
REVIEW_DIR = WORK_BASE / 'review_artifacts' / freeze_git_sha[:12]; REVIEW_DIR.mkdir(parents=True, exist_ok=True)
for p in [FREEZE, RECEIPT]:
    dst = REVIEW_DIR / p.name; shutil.copy2(p, dst); assert sha256(dst) == sha256(p)
if (WORK / 'RUN_CONTRACT.json').exists():
    dst = REVIEW_DIR / 'RUN_CONTRACT.json'; shutil.copy2(WORK / 'RUN_CONTRACT.json', dst); assert sha256(dst) == sha256(WORK / 'RUN_CONTRACT.json')
bundle_manifest = {'schema_version': 't24-review-bundle/1.1', 'task': 'T24', 'capture_source_git_sha': capture_git_sha, 'freeze_source_git_sha': freeze_git_sha, 'deviation_id': 'T24_ENGAGEMENT_MATCHING_2026-08-21', 'small_artifacts': {'source_control_freeze.json': sha256(REVIEW_DIR / 'source_control_freeze.json'), 't24_calibration_capture_receipt.json': sha256(REVIEW_DIR / 't24_calibration_capture_receipt.json')}, 'external_artifacts': {'t24_qwen_calibration.npz': {'path': str(CALIBRATION), 'sha256': sha256(CALIBRATION)}, 'source_control_parameters.npz': {'path': str(PARAMS), 'sha256': sha256(PARAMS)}}, 'claim_boundary': 'T24 freezes calibration/control parameters only; causal specificity requires T29 outcomes.'}
if (REVIEW_DIR / 'RUN_CONTRACT.json').exists(): bundle_manifest['small_artifacts']['RUN_CONTRACT.json'] = sha256(REVIEW_DIR / 'RUN_CONTRACT.json')
BUNDLE_MANIFEST = REVIEW_DIR / 'T24_REVIEW_BUNDLE_MANIFEST.json'; BUNDLE_MANIFEST.write_text(json.dumps(bundle_manifest, indent=2) + '\n', encoding='utf-8')
REVIEW_ZIP = WORK_BASE / f'T24_review_bundle_{freeze_git_sha[:12]}.zip'
with zipfile.ZipFile(REVIEW_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(REVIEW_DIR.iterdir()):
        if p.is_file(): z.write(p, arcname=p.name)
print('PASS: T24 numerical freeze complete and persisted for review')
print('capture source Git SHA:', capture_git_sha); print('freeze source Git SHA:', freeze_git_sha)
print('review dir:', REVIEW_DIR); print('review ZIP:', REVIEW_ZIP); print('review ZIP SHA-256:', sha256(REVIEW_ZIP))
print('calibration NPZ SHA-256:', r['calibration_npz_sha256']); print('controls NPZ SHA-256:', r['numerical_controls_npz_sha256'])
print('Do not start T29 yet. Share the small review ZIP for final verification.')
subprocess.run(['git', '-C', str(PROJECT_DIR), 'status', '--short'])
